# BooksMatch — Système intelligent de recommandation de livres
## IFM30542-H2026 · Groupe 7 · Collège La Cité

Ce projet a pour objectif de construire un système de recommandation de livres capable de :
- proposer des recommandations personnalisées à chaque utilisateur ;
- identifier des livres similaires pour faciliter la découverte.

Notre travail repose sur le dataset **Kaggle Book Recommender System**, qui contient :
- **898 livres**
- **196 296 interactions**
- **66 909 utilisateurs**

Au fil du projet, nous avons conçu une approche **hybride** combinant :
- un modèle collaboratif **SVD biaisé** ;
- une représentation sémantique des livres avec **Jina Embeddings v4 (2048 dimensions)** ;
- un signal de **popularité inspiré d’IMDb**.

La fusion finale repose sur :
- une **z-normalisation** des scores ;
- un **alpha sigmoïdal adaptatif** selon la richesse du profil utilisateur ;
- un **beta optimisé** pour le poids de la popularité ;
- un reranking **MMR sensible à la diversité de genre**.

L’objectif de ce notebook est de montrer, étape par étape, la logique complète du système :  
de la préparation des données jusqu’à l’évaluation, l’interprétation et la sauvegarde des artefacts finaux.

---

## Plan du notebook

| Section | Contenu |
|---|---|
| 1 | Imports et configuration |
| 2 | Chargement des données |
| 3 | Préparation : `ratings_df` et `books_df` |
| 4 | Analyse exploratoire (EDA) |
| 5 | Base livres et score de popularité |
| 6 | Nettoyage textuel et qualité des métadonnées |
| 7 | Génération des embeddings Jina v4 |
| 8 | Similarité cosinus livre-livre |
| 9 | Filtrage K-core |
| 10 | Entraînement et optimisation du modèle SVD |
| 11 | Moteur hybride — profils utilisateurs, alpha, lookups, genre boost |
| 12 | Évaluation Leave-One-Out — optimisation de beta + comparaison de 4 modèles |
| 13 | Métriques complètes — tableaux, matrices et radar |
| 14 | Tests de Wilcoxon + analyse par segment |
| 15 | Système final de recommandation |
| 16 | Tests de cohérence (10/10) |
| 17 | Démonstrations complètes |
| 18 | Modèle auxiliaire supervisé + SHAP |
| 19 | Graphiques de soutenance |
| 20 | Sauvegarde des artefacts |

In [3]:
pip install scikit-surprise

Note: you may need to restart the kernel to use updated packages.


In [5]:
# ================================================================
# IMPORTS ET CONFIGURATION GLOBALE
# Toutes les librairies sont chargées ici une seule fois.
# Les graines aléatoires sont fixées pour la reproductibilité.
# ================================================================

import os, json, pickle, time, warnings, ast, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import requests
import shap

from math import log2
from scipy import stats
from scipy.stats import pearsonr

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split as sk_train_test_split, cross_val_score
from sklearn.inspection import permutation_importance

from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import cross_validate, GridSearchCV, train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

# ── Style graphique ──────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
COLORS = sns.color_palette("muted")
COLOR_CF, COLOR_CB, COLOR_POP, COLOR_HYB, COLOR_ACC = (
    COLORS[0], COLORS[1], COLORS[2], COLORS[3], COLORS[4]
)

# Palette couleurs graphics
C_GREEN  = "#2e7d32"; C_DARK   = "#1b5e20"
C_ORANGE = "#ff6f00"; C_BLUE   = "#1565c0"
C_RED    = "#c62828"; C_GREY   = "#9e9e9e"
C_LIGHT  = "#e8f5e9"; C_BG     = "#fafafa"

plt.rcParams.update({
    "figure.facecolor": C_BG, "axes.facecolor": C_BG,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "serif",
})

print("Imports et configuration prêts.")
print(f"   NumPy  {np.__version__} | Pandas {pd.__version__}")

C:\Users\salim\anaconda3\envs\projet_ds\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports et configuration prêts.
   NumPy  1.26.4 | Pandas 2.3.3


### Chargement initial et premier nettoyage des données

Dans cette étape, nous chargeons les deux sources principales du projet :

- `collab_raw` : contient les interactions entre les utilisateurs et les livres
- `meta_raw` : contient les métadonnées descriptives des livres

Avant toute analyse, nous effectuons un premier nettoyage technique afin de rendre les données plus fiables et plus faciles à exploiter.  
Cela comprend :

- la suppression des colonnes parasites de type `Unnamed`, souvent générées lors des exports CSV ;
- la suppression des doublons dans la table des métadonnées, afin de conserver une fiche unique par livre.

Cette étape permet de poser une base propre pour les analyses exploratoires et les traitements à venir.

In [8]:
# ── Chargement brut des deux sources ─────────────────────────────
collab_raw_brut = pd.read_csv("../data/collaborative_books_df.csv")
meta_raw_brut   = pd.read_csv("../data/collaborative_book_metadata.csv")

print("=" * 70)
print("APERÇU DES DONNÉES BRUTES AVANT NETTOYAGE")
print("=" * 70)

print(f"\ncollab_raw_brut : {collab_raw_brut.shape[0]:,} lignes × {collab_raw_brut.shape[1]} colonnes")
print(f"meta_raw_brut   : {meta_raw_brut.shape[0]:,} lignes × {meta_raw_brut.shape[1]} colonnes")

print(f"\nColonnes collab_raw_brut : {list(collab_raw_brut.columns)}")
print(f"Colonnes meta_raw_brut   : {list(meta_raw_brut.columns)}")

# Vérification des colonnes parasites
unnamed_collab = [c for c in collab_raw_brut.columns if str(c).lower().startswith("unnamed")]
unnamed_meta   = [c for c in meta_raw_brut.columns if str(c).lower().startswith("unnamed")]

print(f"\nColonnes 'Unnamed' dans collab_raw_brut : {unnamed_collab if unnamed_collab else 'Aucune'}")
print(f"Colonnes 'Unnamed' dans meta_raw_brut   : {unnamed_meta if unnamed_meta else 'Aucune'}")

# Vérification rapide des doublons
if "book_id" in meta_raw_brut.columns:
    print(f"\nDoublons sur book_id dans meta_raw_brut : "
          f"{meta_raw_brut.duplicated(subset=['book_id']).sum():,}")

# Aperçu visuel
display(collab_raw_brut.head(3))
display(meta_raw_brut.head(3))

APERÇU DES DONNÉES BRUTES AVANT NETTOYAGE

collab_raw_brut : 196,296 lignes × 7 colonnes
meta_raw_brut   : 96 lignes × 11 colonnes

Colonnes collab_raw_brut : ['Unnamed: 0', 'title', 'book_id', 'user_id_mapping', 'book_id_mapping', 'Predicted Rating', 'Actual Rating']
Colonnes meta_raw_brut   : ['Unnamed: 0', 'book_id', 'title', 'image_url', 'url', 'num_pages', 'ratings_count', 'description', 'genre', 'name', 'book_id_mapping']

Colonnes 'Unnamed' dans collab_raw_brut : ['Unnamed: 0']
Colonnes 'Unnamed' dans meta_raw_brut   : ['Unnamed: 0']

Doublons sur book_id dans meta_raw_brut : 0


,Unnamed: 0,title,book_id,user_id_mapping,book_id_mapping,Predicted Rating,Actual Rating
0,0,I Am the Messenger,19057,1537,299,4.5,5
1,1,I Am the Messenger,19057,23039,299,4.9,3
2,2,I Am the Messenger,19057,39096,299,3.9,3


,Unnamed: 0,book_id,title,image_url,url,num_pages,ratings_count,description,genre,name,book_id_mapping
0,0,5899779,Pride and Prejudice and Zombies Pride and Prejudice and Zombies 1,https://images.gr-assets.com/books/1320449653m/5899779.jpg,https://www.goodreads.com/book/show/5899779-pride-and-prejudice-and-zombies,320,105537,"The New York Times Best Seller is now a major motion picture starring Lily James and Sam Riley, with Matt Smith, Charles Dance, and Lena Headey. \n""It is a truth universally acknowledged that a zo...","['fantasy, paranormal', 'romance', 'fiction', 'history, historical fiction, biography', 'young-adult', 'mystery, thriller, crime']",Jane Austen,808
1,1,872333,Blue Bloods Blue Bloods 1,https://images.gr-assets.com/books/1322281515m/872333.jpg,https://www.goodreads.com/book/show/872333.Blue_Bloods,302,117633,"When the Mayflower set sail in 1620, it carried on board the men and women who would shape America: Miles Standish; John Alden; Constance Hopkins. But some among the Pilgrims were not pure of hear...","['young-adult', 'fantasy, paranormal', 'romance', 'fiction', 'mystery, thriller, crime']",Melissa de la Cruz,217
2,2,15507958,Me Before You Me Before You 1,https://images.gr-assets.com/books/1357108762m/15507958.jpg,https://www.goodreads.com/book/show/15507958-me-before-you,369,609327,"Louisa Clark is an ordinary young woman living an exceedingly ordinary life--steady boyfriend, close family--who has never been farther afield than their tiny village. She takes a badly needed job...","['romance', 'fiction']",Jojo Moyes,385


In [10]:
def drop_unnamed(df):
    """Supprime les colonnes 'Unnamed' générées par certains exports CSV."""
    return df.drop(
        columns=[c for c in df.columns if c.lower().startswith("unnamed")],
        errors="ignore"
    )

# ── Nettoyage initial ────────────────────────────────────────────
collab_raw = drop_unnamed(collab_raw_brut).copy()
meta_raw   = drop_unnamed(meta_raw_brut).copy()

# Suppression des doublons dans les métadonnées
meta_raw = meta_raw.drop_duplicates(subset=["book_id"], keep="first")

print(f"collab_raw : {collab_raw.shape[0]:,} lignes × {collab_raw.shape[1]} colonnes")
print(f"meta_raw   : {meta_raw.shape[0]:,} lignes × {meta_raw.shape[1]} colonnes")

print(f"\nColonnes collab_raw : {list(collab_raw.columns)}")
print(f"Colonnes meta_raw   : {list(meta_raw.columns)}")

collab_raw : 196,296 lignes × 6 colonnes
meta_raw   : 96 lignes × 10 colonnes

Colonnes collab_raw : ['title', 'book_id', 'user_id_mapping', 'book_id_mapping', 'Predicted Rating', 'Actual Rating']
Colonnes meta_raw   : ['book_id', 'title', 'image_url', 'url', 'num_pages', 'ratings_count', 'description', 'genre', 'name', 'book_id_mapping']


In [12]:
def drop_unnamed(df):
    """Supprime les colonnes 'Unnamed' générées par certains exports CSV."""
    return df.drop(
        columns=[c for c in df.columns if c.lower().startswith("unnamed")],
        errors="ignore"
    )

# ── Nettoyage initial ────────────────────────────────────────────
collab_raw = drop_unnamed(collab_raw_brut).copy()
meta_raw   = drop_unnamed(meta_raw_brut).copy()

# Suppression des doublons dans les métadonnées
meta_raw = meta_raw.drop_duplicates(subset=["book_id"], keep="first")

print(f"collab_raw : {collab_raw.shape[0]:,} lignes × {collab_raw.shape[1]} colonnes")
print(f"meta_raw   : {meta_raw.shape[0]:,} lignes × {meta_raw.shape[1]} colonnes")

print(f"\nColonnes collab_raw : {list(collab_raw.columns)}")
print(f"Colonnes meta_raw   : {list(meta_raw.columns)}")

collab_raw : 196,296 lignes × 6 colonnes
meta_raw   : 96 lignes × 10 colonnes

Colonnes collab_raw : ['title', 'book_id', 'user_id_mapping', 'book_id_mapping', 'Predicted Rating', 'Actual Rating']
Colonnes meta_raw   : ['book_id', 'title', 'image_url', 'url', 'num_pages', 'ratings_count', 'description', 'genre', 'name', 'book_id_mapping']


### Interprétation des résultats

Après chargement, nous observons que le jeu de données collaboratif contient **196 296 interactions** réparties sur **6 colonnes**, tandis que le jeu de métadonnées ne contient que **96 lignes** pour **10 colonnes**.

Cette différence de taille est importante : elle montre que nous disposons de beaucoup plus d’informations sur les interactions utilisateur-livre que sur le contenu descriptif des livres.

Les colonnes de `collab_raw` confirment que cette table servira principalement à la recommandation collaborative, grâce aux identifiants utilisateurs, identifiants livres et notes observées.  
De son côté, `meta_raw` contient les informations nécessaires à une approche basée sur le contenu, comme la description, le genre, le nombre de pages ou encore l’auteur.

Ainsi, dès cette première étape, nous identifions une contrainte majeure du projet : la partie collaborative repose sur un volume de données riche, alors que la partie content-based dépend d’un nombre beaucoup plus limité de métadonnées. Ce constat justifie par la suite l’intérêt d’une approche hybride.

### Construction des tables d’interactions et de métadonnées

Dans cette étape, nous préparons les deux tables fondamentales du système de recommandation :

- `ratings_df` : table des interactions utilisateur-livre, qui servira à la modélisation collaborative ;
- `books_df` : table des métadonnées des livres, qui servira à l’enrichissement du catalogue et aux approches basées sur le contenu.

Pour `ratings_df`, nous sélectionnons les colonnes utiles, renommons les variables principales, convertissons les notes au bon format et supprimons les lignes incomplètes. Nous traitons également les doublons utilisateur-livre en agrégeant les notes par moyenne afin de conserver une interaction unique par paire.

Pour `books_df`, nous retenons les variables descriptives importantes telles que le titre, le genre, la description, l’auteur, le nombre de pages ou encore le nombre d’évaluations. Une attention particulière est portée à la colonne `genre`, qui est nettoyée et transformée en une structure plus exploitable.

Enfin, nous réalisons une première vérification de cohérence entre les deux sources de données, notamment en mesurant la couverture des métadonnées et en comparant les titres associés aux mêmes livres.

In [16]:
# ── 2.1 ratings_df ──────────────────────────────────────────────
ratings_df = collab_raw[["user_id_mapping", "book_id_mapping",
                          "book_id", "title", "Actual Rating"]].copy()
ratings_df = ratings_df.rename(columns={
    "user_id_mapping": "user_id",
    "Actual Rating":   "rating"
})
ratings_df["rating"] = pd.to_numeric(ratings_df["rating"], errors="coerce")
ratings_df["title"]  = ratings_df["title"].astype(str)
ratings_df = ratings_df.dropna(subset=["user_id","book_id_mapping","book_id","rating"])

# Moyenne des notes pour les paires dupliquées
ratings_df = ratings_df.groupby(["user_id","book_id_mapping"]).agg({
    "book_id": "last", "title": "last", "rating": "mean"
}).reset_index()
for c in ["user_id","book_id_mapping","book_id"]:
    ratings_df[c] = pd.to_numeric(ratings_df[c], errors="coerce").astype(int)
ratings_df = ratings_df.dropna(subset=["user_id","book_id_mapping","book_id","rating"])

# ── 2.2 books_df ─────────────────────────────────────────────────
def parse_genre(x):
    """
    Convertit la colonne genre (string sérialisé) en liste de tags atomiques.
    Les genres composés comme 'mystery, thriller, crime' sont découpés
    sur la virgule interne pour obtenir des tags individuellement exploitables.
    """
    if pd.isna(x): return []
    if isinstance(x, list): return x
    if isinstance(x, str):
        x = x.strip()
        if not x: return []
        try:
            p = ast.literal_eval(x)
            if isinstance(p, list):
                result = []
                for g in p:
                    for sub in str(g).split(","):
                        sub = sub.strip()
                        if sub: result.append(sub)
                return result
            return [x]
        except Exception as e:
            warnings.warn(f"Parsing genre impossible : '{x}' → {e}")
            return [s.strip() for s in x.split(",") if s.strip()]
    return []

books_df = meta_raw[["book_id","title","genre","description",
                      "num_pages","ratings_count","image_url","url","name"]].copy()
books_df = books_df.rename(columns={"name": "author"})
books_df["author"] = books_df["author"].fillna("unknown")

for c in ["title","genre","description"]:
    books_df[c] = books_df[c].fillna("").astype(str)

books_df["book_id"]       = pd.to_numeric(books_df["book_id"], errors="coerce")
books_df["num_pages"]     = pd.to_numeric(books_df["num_pages"], errors="coerce")
books_df["ratings_count"] = pd.to_numeric(books_df["ratings_count"], errors="coerce")
books_df = books_df.dropna(subset=["book_id"])
books_df["book_id"]       = books_df["book_id"].astype(int)

books_df["genre_list"]  = books_df["genre"].apply(parse_genre)
books_df["genre_clean"] = books_df["genre_list"].apply(lambda l: " | ".join(l))

# ── Vérification cohérence titres ────────────────────────────────
cov = (len(set(ratings_df["book_id"].unique()) & set(books_df["book_id"].unique()))
       / ratings_df["book_id"].nunique())

check_titles = books_df[["book_id","title"]].merge(
    ratings_df[["book_id","title"]].drop_duplicates("book_id"),
    on="book_id", suffixes=("_meta","_collab")
)
n_mismatch = (check_titles["title_meta"] != check_titles["title_collab"]).sum()

print(f"ratings_df : {len(ratings_df):,} interactions | "
      f"{ratings_df['user_id'].nunique():,} users | "
      f"{ratings_df['book_id_mapping'].nunique()} livres")
print(f"books_df   : {len(books_df)} livres avec metadata")
print(f"Couverture metadata : {cov:.2%}")
print(f"Désaccords de titres : {n_mismatch} / {len(check_titles)}")

ratings_df : 196,296 interactions | 66,909 users | 898 livres
books_df   : 96 livres avec metadata
Couverture metadata : 10.69%
Désaccords de titres : 0 / 96


### Interprétation des résultats

À l’issue de cette étape, nous obtenons deux tables propres et complémentaires.

La table `ratings_df` contient **196 296 interactions**, réparties sur **66 909 utilisateurs** et **898 livres**. Après agrégation des doublons, chaque ligne correspond à une paire unique `(utilisateur, livre)` associée à une note exploitable. Cette structure constitue la base de la future matrice d’interactions pour la recommandation collaborative.

La table `books_df`, quant à elle, ne contient que **96 livres avec métadonnées**. La couverture des métadonnées est donc de **10,69 %**, ce qui signifie que seule une faible proportion des livres présents dans les interactions dispose également d’informations descriptives exploitables.

Ce résultat met en évidence une contrainte importante du projet : la partie collaborative repose sur un volume de données riche, alors que la partie content-based est limitée par une faible couverture des métadonnées. Ce constat justifie le recours à une approche hybride plutôt qu’à un modèle uniquement basé sur le contenu.

Enfin, la vérification de cohérence entre les deux sources montre **0 désaccord de titres sur 96 livres communs**, ce qui confirme une bonne qualité d’alignement pour les livres disposant de métadonnées.